In [1]:
import pandas as pd
import datetime as dt
from acled import Acled

In [2]:
acled = Acled()

INFO:acled:Access token correctly retrieved.


Train: Data from January 2018 to December 2022. This includes the 2018 Sudanese revolution but excludes the 2023 Civil War.
Test onset civil war: Data from January 2023 to December 2023 which includes the escalation of the civil war.
Test active civil war: Data from January 2024 to December 2025 which includes fluctuations in ongoing civil war.


In [3]:
countries = ["Sudan"]
start_date = "2017-07-01"
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date ="2023-12-31"

active_start_date = "2024-01-01"
active_end_date ="2024-12-31"

In [4]:
all_data = acled.get_data(countries, train_start_date, train_end_date)

INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:All data successfully fetched.


In [5]:
# train = acled.get_data(countries, train_start_date, train_end_date)
# train.to_csv("../data/train.csv", index=False)
#
# onset = acled.get_data(countries, onset_start_date, onset_end_date)
# onset.to_csv("../data/onset.csv", index=False)

# active = acled.get_data(countries, active_start_date, active_end_date)
# active.to_csv("../active.csv", index=False)


In [6]:
def map_events(df: pd.DataFrame) -> pd.DataFrame:
    # 1 = Conflict event (Y)
    # 0 = Non-conflict (used for features)

    acled_subevent_mapping = {
        # BATTLES (Conflict)
        "Armed clash": 1,
        "Government regains territory": 1,
        "Non-state actor overtakes territory": 1,

        # EXPLOSIONS / REMOTE VIOLENCE (Conflict)
        "Air/drone strike": 1,
        "Chemical weapon": 1,
        "Remote explosive/landmine/IED": 1,
        "Shelling/artillery/missile attack": 1,
        "Suicide bomb": 1,
        "Grenade": 1,

        # VIOLENCE AGAINST CIVILIANS (Conflict)
        "Abduction/forced disappearance": 1,
        "Attack": 1,
        "Sexual violence": 1,

        # RIOTS (Conflict)
        "Mob violence": 1,
        "Violent demonstration": 1,

        # PROTESTS (Non-conflict)
        "Excessive force against protesters": 0,
        "Peaceful protest": 0,
        "Protest with intervention": 0,

        # STRATEGIC DEVELOPMENTS (Non-conflict)
        "Agreement": 0,
        "Arrests": 0,
        "Change to group/activity": 0,
        "Disrupted weapons use": 0,
        "Headquarters or base established": 0,
        "Looting/property destruction": 0,
        "Non-violent transfer of territory": 0,
        "Other": 0
    }
    df["conflict"] = df["sub_event_type"].apply(lambda x: acled_subevent_mapping[x])
    return df


In [7]:
def conflict_regional_monthly(df):
    df = df.copy()
    df = df[df["conflict"] == 1]
    df['year_month'] = df['event_date'].dt.to_period('M')

    df_grouped = df.groupby(["admin2", "year_month"])["event_id_cnty"].count().reset_index(name="event_count")
    df_grouped = df_grouped.sort_values(by=['admin2', 'year_month'])

    df_grouped['rolling_mean_6m'] = df_grouped.groupby('admin2')['event_count'].transform(
        lambda x: x.rolling(window=6, min_periods=6).mean().shift(1)
    )

    df_grouped['rolling_std_6m'] = df_grouped.groupby('admin2')['event_count'].transform(
        lambda x: x.rolling(window=6, min_periods=6).std().shift(1)
    )

    return df_grouped

In [8]:
df = map_events(all_data)
df_grouped = conflict_regional_monthly(df)

In [11]:
train_df = df_grouped[
    (df_grouped['year_month'] >= train_start_date) &
    (df_grouped['year_month'] <= train_end_date)
].copy()

# Test Set 1: Civil War Escalation (2023)
onset_test_df = df_grouped[
    (df_grouped['year_month'] >= onset_start_date) &
    (df_grouped['year_month'] <= onset_end_date)
].copy()

# Test Set 2: Active Conflict (2024)
active_test_df = df_grouped[
    (df_grouped['year_month'] >= active_start_date) &
    (df_grouped['year_month'] <= active_end_date)
].copy()

In [12]:
train_df

,admin2,year_month,event_count,rolling_mean_6m,rolling_std_6m
0,Abassiya,2018-06,1,NaN,NaN
1,Abassiya,2018-09,1,NaN,NaN
2,Abassiya,2018-12,1,NaN,NaN
3,Abassiya,2019-09,2,NaN,NaN
4,Abassiya,2019-10,1,NaN,NaN
...,...,...,...,...,...
1520,Zalingi,2022-05,1,1.666667,0.816497
1521,Zalingi,2022-06,1,1.333333,0.516398
1522,Zalingi,2022-08,1,1.333333,0.516398
1523,Zalingi,2022-11,3,1.333333,0.516398
